In [ ]:
from cube_nn import CubeValueResNet
import torch
from cube import Cube
from solvers import AStarSolver
from cube_nn import NNValueFunctionType
from autotuner import autotune_parameters
from tqdm import tqdm

In [ ]:
# Load NN value function
I = 6000
net = CubeValueResNet()
net.load_state_dict(torch.load(f'temp_models/resnet2/cube_value_resnet_iter_{I}.pth'))

# Create a solver using the NN value function
net.set_value_function_type(NNValueFunctionType.STANDARD)
solver = AStarSolver(net.as_value_function(), weight=0.18, noise=0.15, max_moves=40, max_queue_size=1000000, t_max=20, seed=42)

In [3]:
# Generate cubes to evaluate and tune the solver on
N = 100
scramble_moves = 50
cubes = [Cube(n_scramble_moves=scramble_moves) for _ in range(N)]

# Define evaluation function
def evaluate_solver_on_cubes(weight: float, noise: float) -> float:
    solver._weight = weight
    solver._noise = noise
    solved_cubes = 0
    for cube in tqdm(cubes, desc=f'Evaluating weight={weight}, noise={noise}', leave=False):
        solution = solver(cube)
        if solution is not None:
            solved_cubes += 1
    print(f'Weight: {weight}, Noise: {noise}, Solved: {solved_cubes}/{N}')
    return solved_cubes / N  # Return fraction of cubes solved

# Autotune parameters
initial_values = {'weight': 0.18, 'noise': 0.15}
bounds = {'weight': (0.15, 0.2), 'noise': (0.12, 0.18)}

best_params = autotune_parameters(evaluate_solver_on_cubes, initial_values, bounds, step_size=0.01)

Weight: 0.18, Noise: 0.15, Solved: 62/100


Weight: 0.15, Noise: 0.15, Solved: 62/100


Weight: 0.16, Noise: 0.15, Solved: 67/100


Weight: 0.17, Noise: 0.15, Solved: 63/100


Weight: 0.18000000000000002, Noise: 0.15, Solved: 63/100


...

Weight: 0.19000000000000003, Noise: 0.13, Solved: 65/100


Weight: 0.19000000000000003, Noise: 0.14, Solved: 64/100


Weight: 0.19000000000000003, Noise: 0.15000000000000002, Solved: 74/100


Weight: 0.19000000000000003, Noise: 0.16000000000000003, Solved: 68/100


Weight: 0.19000000000000003, Noise: 0.17000000000000004, Solved: 56/100
Autotuning completed in 1 iterations.
Tuned parameters: {'weight': 0.19000000000000003, 'noise': 0.15000000000000002}


In [ ]:
tuned_solver = AStarSolver(net.as_value_function(), weight=0.19, noise=0.15,
                            max_moves=40, max_queue_size=1000000, t_max=20, seed=42)